# 5주차 ① 퍼셉트론에서 MLP까지 — 실습 1~4  〔빈칸본〕

> **1교시 노트북입니다. 데이터를 내려받지 않아도 됩니다.**
> 빈칸은 모두 **7곳 · 13줄** 입니다 — 실습 1에 4곳(7줄), 실습 2에 1곳(2줄),
> 실습 3에 2곳(4줄). **실습 4는 빈칸이 없고, 주석(`#`) 2줄을 지우고 실행만 합니다.**
>
> 빈칸마다 **힌트**와 **확인**(채우고 나면 이런 값이 나온다)이 붙어 있습니다.
> 막히면 확인 줄을 먼저 보세요. 다 채운 노트북은
> `06_xor_and_activations.ipynb` 로 저장해 제출합니다.

> **정답은 각 실습 셀 바로 아래 `✅ 정답 보기` 를 눌러 펼칠 수 있습니다.**
> 먼저 스스로 채워 보고, 막혔을 때만 펼치세요.

> **오늘 기억할 것**: 오늘 배우는 것은 수식이 아니라 **"돌려 보고 숫자로 확인하는 습관"** 입니다.

---

## 실습 1 — 퍼셉트론의 한계를 직접 확인한다 ★★

**오늘 수업 전체의 출발점입니다.**

XOR 은 *"두 입력이 서로 다를 때만 1"* 인 아주 간단한 규칙입니다. 점이 네 개뿐입니다.
그런데 이 네 점을 **직선 하나로는 못 나눕니다.**

세 가지 모델을 **똑같은 방법으로** 학습시켜 보고, 나온 숫자를 비교합니다.

| | 모델 | 예상 |
|---|---|---|
| ⓐ | 층 하나 | ? |
| ⓑ | 층 셋 — **사이에 활성함수 없음** | ? |
| ⓒ | 층 둘 — 사이에 **ReLU 한 줄** | ? |

> **먼저 찍어 보세요.** ⓑ 는 층이 ⓒ 보다 많습니다. 어느 쪽이 잘할까요?

In [1]:
# 셀 1 — 준비와 XOR 데이터
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "Malgun Gothic"
plt.rcParams["axes.unicode_minus"] = False

# XOR :  (0,0) -> 0    (0,1) -> 1    (1,0) -> 1    (1,1) -> 0
# ───── 실습 1 · 빈칸 ① (2줄) ─────
# 힌트 : 입력 4개를 (4,2) 텐서로, 정답 4개를 (4,1) 텐서로. 소수점(0. 처럼)을 꼭 붙인다
# 확인 : X 는 torch.Size([4, 2]), Y 는 torch.Size([4, 1])



print("입력 X :", X.shape)
print("정답 Y :", Y.shape)

plt.figure(figsize=(3.4, 3.4))
plt.scatter(X[:, 0], X[:, 1], c=Y.squeeze(), cmap="coolwarm", s=320, edgecolors="k")
plt.title("XOR — 직선 하나로 색을 나눌 수 있을까?")
plt.xlabel("x1"); plt.ylabel("x2"); plt.grid(alpha=0.3)
plt.show()

NameError: name 'X' is not defined

<details>
<summary>✅ <b>정답 보기</b> — 실습 1 · 빈칸 ① (2줄)</summary>

```python
X = torch.tensor([[0., 0.], [0., 1.], [1., 0.], [1., 1.]])
Y = torch.tensor([[0.], [1.], [1.], [0.]])
```

`0.` 처럼 소수점을 붙여야 float 텐서가 됩니다. 정수로 쓰면 뒤에서 `BCEWithLogitsLoss` 가 타입 오류를 냅니다.

</details>

> **관찰 포인트**: 빨강 둘과 파랑 둘이 **대각선으로 마주** 보고 있습니다.
> 자를 하나 놓고 아무리 돌려 봐도 같은 색끼리 한쪽으로 모을 수 없습니다.

In [ ]:
# 셀 2 — 세 실험이 공통으로 쓸 학습 함수 (그대로 실행하세요)
def train_xor(model, steps=3000, lr=0.1):
    """XOR 를 steps 번 학습하고 (마지막 손실, 네 점의 확률) 을 돌려준다."""
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.BCEWithLogitsLoss()   # 이 안에 시그모이드가 들어 있다 (출력용, 층 사이 활성함수가 아니다)

    for _ in range(steps):
        pred = model(X)                  # ① 순전파
        loss = loss_fn(pred, Y)          # ② 손실
        opt.zero_grad()                  # ③ 지우기
        loss.backward()                  # ④ 역전파
        opt.step()                       # ⑤ 갱신

    prob = torch.sigmoid(model(X)).squeeze()   # 확률로 보려고 한 번 더 — 역시 출력용
    return loss.item(), prob


def report(name, loss, prob):
    p = "  ".join(f"{v:.3f}" for v in prob.tolist())
    print(f"{name:20s} | 손실 {loss:.4f} | 확률  {p}")


print("※ 안쪽 5줄이 4주차의 학습 루프 그대로입니다. 오늘 3교시에 다시 만납니다.")

> **잠깐 — 위 함수 안의 5줄을 보세요.** 순전파 → 손실 → 지우기 → 역전파 → 갱신.
> **4주차에 쓴 것과 글자 그대로 같습니다.** 오늘 새로 배우는 건 루프가 아니라 **모델의 모양**입니다.

### ⓐ 층 하나로 풀어 본다

In [2]:
# 셀 3 — ⓐ 층 하나 — 4주차 로지스틱 회귀와 같은 모델
torch.manual_seed(0)

# ───── 실습 1 · 빈칸 ② (1줄) ─────
# 힌트 : 입력 2개 -> 출력 1개인 층 하나를 만든다. nn.Linear(?, ?)
# 확인 : 손실이 0.6931 근처에서 멈춘다


loss_a, prob_a = train_xor(model_a)
report("ⓐ 1층", loss_a, prob_a)

NameError: name 'train_xor' is not defined

<details>
<summary>✅ <b>정답 보기</b> — 실습 1 · 빈칸 ② (1줄)</summary>

```python
model_a = nn.Linear(2, 1)
```

입력 2개 → 출력 1개. 파라미터는 `w` 2개 + `b` 1개 = **3개**뿐입니다.

</details>

> **관찰 포인트 ★**: 확률 네 개가 전부 **0.500** 입니다. *"모르겠다, 반반이다"* 라는 뜻입니다.
> 손실 **0.6931** 은 **완전히 모를 때 나오는 값**입니다. 3000번을 돌렸는데 여기서 꿈쩍도 안 했습니다.
>
> 정답은 `0 1 1 0` 인데 모델은 `0.5 0.5 0.5 0.5` 를 내놓았습니다. **학습이 안 된 것**입니다.

### ⓑ 그럼 층을 늘리면 되지 않을까

In [3]:
# 셀 4 — ⓑ 층을 셋으로 늘린다 (층 사이에 활성함수 없음)
torch.manual_seed(0)

model_b = nn.Sequential(
    nn.Linear(2, 8),
# ───── 실습 1 · 빈칸 ③ (2줄) ─────
# 힌트 : 가운데 8 -> 8 층과 마지막 8 -> 1 층을 한 줄씩. 활성함수는 넣지 않는다
# 확인 : ⓐ 와 똑같이 손실 0.6931, 확률 0.500


)

loss_b, prob_b = train_xor(model_b)
report("ⓑ 3층(활성함수 없음)", loss_b, prob_b)

NameError: name 'train_xor' is not defined

<details>
<summary>✅ <b>정답 보기</b> — 실습 1 · 빈칸 ③ (2줄)</summary>

```python
    nn.Linear(8, 8),
    nn.Linear(8, 1),
```

`nn.Sequential(` 안쪽이므로 **4칸 들여쓰기**하고 줄 끝에 쉼표를 붙입니다. 활성함수는 일부러 넣지 않습니다.

</details>

> **결과 해석 ★★ (오늘 가장 중요한 한 장면)**
>
> 층을 **3배로 늘렸는데 결과가 ⓐ 와 똑같습니다.** 손실 0.6931, 확률 0.500.
> 파라미터는 3개에서 89개로 늘었는데 **아무것도 나아지지 않았습니다.**
>
> **층을 쌓는 것 자체는 아무 힘이 없습니다.** 층을 그냥 이어 붙이면
> 아무리 많이 쌓아도 **결국 층 하나와 똑같은 일**을 합니다.

### ⓒ 층은 줄이고, 대신 사이에 ReLU 한 줄

In [4]:
# 셀 5 — ⓒ 층 둘 + ReLU 한 줄
torch.manual_seed(0)

model_c = nn.Sequential(
    nn.Linear(2, 8),
# ───── 실습 1 · 빈칸 ④ (2줄) ─────
# 힌트 : ReLU 한 줄과 8 -> 1 층 한 줄. 순서에 주의
# 확인 : 손실이 0.003 근처까지 떨어진다


)

loss_c, prob_c = train_xor(model_c)
report("ⓒ 2층 + ReLU", loss_c, prob_c)

NameError: name 'train_xor' is not defined

<details>
<summary>✅ <b>정답 보기</b> — 실습 1 · 빈칸 ④ (2줄)</summary>

```python
    nn.ReLU(),
    nn.Linear(8, 1),
```

순서가 핵심입니다 — **`Linear` → `ReLU` → `Linear`**. ⓑ 와 다른 것은 `nn.ReLU()` 한 줄뿐인데 손실이 0.6931 에서 0.003 으로 떨어집니다.

</details>

> **결과 해석 ★★**: 손실이 **0.6931 → 0.003**. 확률은 **0.003 / 0.996 / 0.996 / 0.003**
> 으로 정답 `0 1 1 0` 을 거의 정확히 맞혔습니다.
>
> **ⓑ 보다 층이 적은데 ⓒ 가 풀었습니다.** 일을 한 것은 층이 아니라 **ReLU 한 줄**입니다.

In [5]:
# 셀 6 — 셋을 나란히 놓고 본다
print("정답                 |             | 확률  0.000  1.000  1.000  0.000")
print("-" * 64)
report("ⓐ 1층", loss_a, prob_a)
report("ⓑ 3층(활성함수 없음)", loss_b, prob_b)
report("ⓒ 2층 + ReLU", loss_c, prob_c)

# 세 모델이 평면을 어떻게 나누는지 그림으로 확인한다
gx, gy = torch.meshgrid(torch.linspace(-0.3, 1.3, 120),
                        torch.linspace(-0.3, 1.3, 120), indexing="ij")
grid = torch.stack([gx.reshape(-1), gy.reshape(-1)], dim=1)

fig, ax = plt.subplots(1, 3, figsize=(12, 3.8))
for a, m, t in zip(ax, [model_a, model_b, model_c],
                   ["ⓐ 1층", "ⓑ 3층 (활성함수 없음)", "ⓒ 2층 + ReLU"]):
    with torch.no_grad():
        z = torch.sigmoid(m(grid)).reshape(gx.shape)
    a.contourf(gx, gy, z, levels=20, cmap="coolwarm", vmin=0, vmax=1)
    a.scatter(X[:, 0], X[:, 1], c=Y.squeeze(), cmap="coolwarm", s=220, edgecolors="k")
    a.set_title(t)
plt.tight_layout()
plt.show()

정답                 |             | 확률  0.000  1.000  1.000  0.000
----------------------------------------------------------------


NameError: name 'report' is not defined

> **관찰 포인트 ★★**: 왼쪽 두 그림은 **색이 거의 변하지 않는 밋밋한 면**입니다 —
> 어디를 봐도 0.5. 오른쪽만 **네 귀퉁이가 제대로 갈라져** 있습니다.
>
> ### 오늘의 첫 결론
>
> **층 사이에 활성함수가 없으면, 층을 아무리 쌓아도 층 하나와 같다.**
> 그래서 층을 쌓을 때는 **반드시 사이에 활성함수를 넣는다.**
>
> *(왜 그런지 식으로 확인하고 싶다면 교수님께 `📘 이론 보충` 을 요청하세요.
> 식 없이도 위 실험이 같은 이야기를 합니다.)*

---

## 잠깐 — 다층 신경망(MLP)의 구조

층을 쌓을 때는 **사이에 활성함수를 넣는다**는 것을 방금 확인했습니다.
**그럼 진짜 신경망은 어떻게 생겼을까요?** 오늘 오후에 만들 모델을 미리 봅니다.

```
   입력층        은닉층 1        은닉층 2        출력층
   (784)          (256)           (128)          (10)
    ○ ─┐          ○               ○              ○  티셔츠
    ○ ─┼──ReLU──→ ○ ──ReLU────→  ○ ─────────→   ○  바지
    ○ ─┤          ○               ○              ○  ...
    ⋮  │          ⋮               ⋮              ○  앵클부츠
    ○ ─┘          ○               ○
       W1, b1         W2, b2         W3, b3

   파라미터 수 = 784×256 + 256 + 256×128 + 128 + 128×10 + 10 = 235,146 개
```

| 층 | 무엇 | 개수 |
|---|---|---|
| **입력층** | 28×28 그림을 한 줄로 편 것 | 784 |
| **은닉층 1·2** | 뒤에 **ReLU 를 붙인다** ★ | 256 · 128 |
| **출력층** | 옷 10종류. **활성함수를 붙이지 않는다** ★ | 10 |

> **관찰 포인트 ★**: 4주차에는 파라미터가 **2개**였습니다. 오늘은 **23만 개**입니다.
> 그래서 `w -= lr * w.grad` 를 손으로 쓸 수 없고, `nn.Module` 과 `optimizer` 가 필요합니다.
>
> **위 계산식은 외우지 마세요.** 2교시에 모델을 만들고 나서
> `sum(p.numel() for p in model.parameters())` 한 줄로 **235,146** 이 나오는 것을 직접 확인합니다.
>
> **함정 ★**: 은닉층 뒤에는 ReLU 를 붙이고 **출력층 뒤에는 아무것도 붙이지 않습니다.**
> 이유는 실습 4에서 숫자로 확인합니다.

---

## 실습 2 — 활성함수 3종의 모양

방금 ReLU 하나로 문제가 풀렸습니다. 활성함수에는 어떤 것들이 있고 각각 어디에 쓸까요?
**식을 외우지 말고 모양을 기억하세요.**

In [ ]:
# 셀 7 — 활성함수 3종의 모양
z = torch.linspace(-6, 6, 200)
logits = torch.tensor([2.0, 1.0, 0.1, -1.0, 0.5])    # 제공 — 그대로 쓰세요

fig, ax = plt.subplots(1, 3, figsize=(13, 3.5))

# ───── 실습 2 · 빈칸 ① (2줄) — `______` 자리에 함수 호출만 채우세요 ─────
# 힌트 : ① z 를 sigmoid 에 통과시킨 값   ② logits 를 softmax 에 통과시킨 값 (dim=0)
# 확인 : 가운데는 S 자 곡선, 오른쪽은 막대 5개
#        그림을 그리는 코드(plot·bar·제목)는 이미 채워져 있으니 손대지 않습니다

ax[0].plot(z, torch.relu(z))          # ← 예시로 채워 두었습니다. 아래 두 줄도 이렇게 쓰면 됩니다
ax[1].plot(z, ______)                 # ← 빈칸 ①-1
ax[2].bar(range(5), ______)           # ← 빈칸 ①-2

ax[0].set_title("ReLU — 음수는 0, 양수는 그대로")
ax[1].set_title("Sigmoid — 0 과 1 사이")
ax[2].set_title("Softmax — 다 더하면 1")
for a in ax[:2]:
    a.axhline(0, color="gray", lw=0.5)
    a.axvline(0, color="gray", lw=0.5)
plt.tight_layout()
plt.show()

NameError: name '______' is not defined

<details>
<summary>✅ <b>정답 보기</b> — 실습 2 · 빈칸 ① (2줄)</summary>

```python
ax[1].plot(z, torch.sigmoid(z))                    # 빈칸 ①-1
ax[2].bar(range(5), torch.softmax(logits, dim=0))  # 빈칸 ①-2
```

`______` 자리에 함수 호출만 넣으면 됩니다. Softmax 는 `dim=0` — 5개 원소를 묶어 합이 1 이 되게 합니다.

</details>

| 활성함수 | 어떤 모양인가 | 어디에 쓰나 |
|---|---|---|
| **ReLU** | 음수는 0 으로 눌러 버리고, 양수는 그대로 통과 | **은닉층의 기본** ★ 오늘 우리가 쓸 것 |
| **Sigmoid** | 부드러운 S 자. 결과가 항상 0 과 1 사이 | 이진 분류의 **출력층** (4주차에 썼다) |
| **Softmax** | 막대 여러 개의 높이를 **합이 1** 이 되게 맞춘다 | **다중 분류의 출력층** |

> **포인트**: ReLU 는 놀랄 만큼 단순합니다 — **음수면 0, 양수면 그대로.**
> 이렇게 단순한 것이 Sigmoid 를 이긴 이유를 다음 실습에서 숫자로 봅니다.

In [ ]:
# 셀 8 — Softmax 는 정말 "확률" 인가
p = torch.softmax(logits, dim=0)

print("logits (날것)  :", [round(v, 2) for v in logits.tolist()])
print("softmax 결과   :", [round(v, 4) for v in p.tolist()])
print("전부 더하면    :", round(p.sum().item(), 6))
print("가장 큰 것     :", p.argmax().item(), "번")

> **관찰 포인트 ★**: 합이 **1.0** 입니다. 그래서 *"이 이미지가 티셔츠일 확률 0.56,
> 바지일 확률 0.21…"* 처럼 읽을 수 있습니다.
> 4주차의 **Sigmoid 는 두 부류**, 오늘의 **Softmax 는 여러 부류**용입니다.

---

## Sigmoid 의 약점 — 기울기를 곱하며 뒤로 간다 ★

```
   Sigmoid                      그 기울기(미분)
    1 ┤    ___------             0.25 ┤    ／＼
      │  _/                           │   /   ＼
  0.5 ┤─●───────                      │  /     ＼
      │_/                          0  ┤_/       ＼____
      └────────── z                   └──────────────── z
                                        ↑ 양끝에서 거의 0 ★
```

Sigmoid 의 기울기는 **가장 클 때도 0.25** 이고, 양끝으로 가면 거의 0 입니다.

> ### 핵심 질문 ★★
>
> 역전파는 각 층의 기울기를 **곱하면서** 뒤로 갑니다.
> 층이 5개면 기울기를 5번 곱하는 셈입니다. 그럼 —
>
> ```
>    0.2 × 0.2 × 0.2 × 0.2 × 0.2 = ?
>
>                              → 0.00032
> ```
>
> **1만분의 3** 입니다. 앞쪽 층에 도달할 때쯤 기울기가 **사실상 0** 이 됩니다.

1 보다 작은 수를 계속 곱하면 순식간에 0 에 붙습니다.
**그럼 진짜로 그렇게 되는지, 지금 직접 재 봅시다.**

---

## 실습 3 — 깊게 쌓으면 앞쪽 층은 어떻게 되나 ★

*"ReLU 를 쓴다"* 를 외우지 말고 **이유를 직접 재 봅시다.**

**첫 층의 기울기 ÷ 끝 층의 기울기** 를 잽니다.

- 이 값이 **1 에 가까우면** 앞쪽까지 신호가 잘 갔다는 뜻
- 이 값이 **0 에 가까우면** 앞쪽 층은 거의 학습되지 않는다는 뜻

**ReLU 는 어떨까요?** 미리 답하지 않겠습니다. 아래 표가 답합니다.

In [ ]:
# 셀 9 — 층을 깊게 하면 앞쪽 기울기가 어떻게 되나
def make_net(act, n_hidden):
    layers = []
    for _ in range(n_hidden):
# ───── 실습 3 · 빈칸 ① (1줄) ─────
# 힌트 : 은닉층 하나 = nn.Linear(64, 64) 와 활성함수 act() 를 layers 에 이어 붙인다
# 확인 : 아래 표가 3행으로 출력되면 성공

    layers += [nn.Linear(64, 1)]
    return nn.Sequential(*layers)


torch.manual_seed(0)                 # 누구 PC 에서나 같은 숫자가 나오도록
x = torch.randn(32, 64)
y = torch.randn(32, 1)

print(f"{'은닉층':>6} | {'Sigmoid':>12} | {'ReLU':>12}    ← 첫 층 기울기 ÷ 끝 층 기울기")
print("-" * 56)

for n in [3, 5, 7]:
    ratio = []
    for act in [nn.Sigmoid, nn.ReLU]:
        torch.manual_seed(0)
        net = make_net(act, n)
        loss = ((net(x) - y) ** 2).mean()
        loss.backward()

# ───── 실습 3 · 빈칸 ② (3줄) ─────
# 힌트 : 맨 앞 층은 net[0], 맨 뒤 층은 net[-1]. weight.grad 의 절댓값 평균을 구해 비율을 ratio 에 담는다
# 확인 : Sigmoid 열이 7.4e-04 -> 9.6e-06 -> 2.4e-07 로 줄어든다




    print(f"{n:>6} | {ratio[0]:>12.2e} | {ratio[1]:>12.2e}")

<details>
<summary>✅ <b>정답 보기</b> — 실습 3 · 빈칸 ① (1줄) · ② (3줄)</summary>

```python
# 빈칸 ① — for _ in range(n_hidden): 안쪽 (8칸 들여쓰기)
        layers += [nn.Linear(64, 64), act()]

# 빈칸 ② — loss.backward() 다음 (8칸 들여쓰기)
        first = net[0].weight.grad.abs().mean().item()
        last = net[-1].weight.grad.abs().mean().item()
        ratio.append(first / last)
```

`net[0]` 은 맨 앞 층, `net[-1]` 은 맨 뒤 층입니다. `.abs().mean()` 으로 기울기의 크기만 보고, 그 비율을 `ratio` 에 담습니다. **들여쓰기가 틀리면 표가 3행으로 안 나옵니다.**

</details>

> **결과 해석 ★★ (중간고사 출제 지점)**
>
> | 은닉층 | 3개 | 5개 | 7개 |
> |---|---|---|---|
> | **Sigmoid** | 0.00074 | 0.0000096 | **0.00000024** |
> | **ReLU** | 0.077 | 0.025 | **0.043** |
>
> 바로 위에서 `0.2 × 0.2 × 0.2 × 0.2 × 0.2 = 0.00032` 라고 계산했죠.
> **실제로 재 보니 은닉층 5개에서 0.0000096** 으로, 예상보다 더 작습니다 —
> 양끝에 걸린 뉴런이 많아 0.2 보다 작은 값이 곱해졌기 때문입니다.
>
> **Sigmoid 쪽을 보세요.** 층이 둘 늘 때마다 **수십 배씩** 작아집니다.
> 7층에서는 첫 층 기울기가 끝 층의 **400만 분의 1** 수준입니다.
> 기울기가 이렇게 작으면 그 층은 **사실상 학습되지 않습니다.**
> 앞쪽 층이 안 배우면 깊게 쌓은 의미가 없습니다 — 이것을 **기울기 소실** 이라고 합니다.
>
> **ReLU 쪽은 층을 늘려도 0.02 ~ 0.08 사이에 머뭅니다.** 줄어들기는커녕 층 7개에서
> 오히려 조금 올랐습니다. 두 열의 자릿수를 비교해 보세요 — 7층에서 **10만 배 차이**입니다.
>
> **그래서 은닉층에는 ReLU 를 씁니다.** 외울 것은 이 한 줄이고, 근거는 위 표입니다.

| 위치 | 무엇을 쓰나 | 왜 |
|---|---|---|
| **은닉층** | **ReLU** | 깊어져도 앞쪽까지 신호가 간다 |
| 출력층 (둘 중 하나) | Sigmoid | 확률 하나가 필요 |
| 출력층 (여럿 중 하나) | Softmax | 클래스별 확률이 필요 |

> **포인트**: 출력층에서는 Sigmoid·Softmax 를 여전히 씁니다.
> **거기서는 뒤로 더 쌓지 않으므로** 기울기 소실이 문제되지 않습니다.

---

## 실습 4 — 10개 중 하나를 고르는 문제의 손실

4주차에는 **MSE**(값 맞히기)와 **BCE**(둘 중 하나)를 썼습니다.
오늘 오후에 풀 문제는 **옷 사진을 10종류 중 하나로 분류**하는 것입니다.

| 손실 | 쓰는 곳 | PyTorch |
|---|---|---|
| **MSE** | 회귀 — 값을 맞힌다 | `nn.MSELoss()` |
| **BCE** | 이진 분류 — 둘 중 하나 | `nn.BCEWithLogitsLoss()` (4주차) |
| **CrossEntropy** | **다중 분류 — 여럿 중 하나** ★ | `nn.CrossEntropyLoss()` (오늘) |

### 순전파·역전파를 4주차와 잇는다

```
   [순전파 →]
   x(784) ─W1,ReLU→ h1(256) ─W2,ReLU→ h2(128) ─W3→ logits(10) ─CE→ loss

   [역전파 ←]
   W1.grad ←──────── W2.grad ←──────── W3.grad ←──────────────── loss.backward()
            ↑ 여기까지 신호가 살아서 오게 하는 것이 ReLU 의 역할
```

> 4주차 계산 그래프 그림과 **구조가 같습니다.** 층이 늘었을 뿐입니다.
> **`backward()` 한 줄이 23만 개 파라미터의 `.grad` 를 전부 채웁니다.**
> 위 그림의 왼쪽 끝 `W1.grad` 까지 신호가 도착하는가 — **실습 3에서 잰 것이 바로 그것**입니다.

**함정이 두 개** 있습니다. 코드로 직접 확인합니다.

In [ ]:
# 셀 10 — CrossEntropyLoss 의 함정 두 가지
logits3 = torch.tensor([[2.0, 1.0, 0.1]])    # 배치 1개, 클래스 3개
target = torch.tensor([0])                   # ★ 함정 1 : 정답은 "클래스 번호". [1,0,0] 이 아니다
loss_fn = nn.CrossEntropyLoss()

# ───── 실습 4 — 타이핑 없습니다. 아래 두 줄의 맨 앞 `#` 만 지우고 실행하세요 ─────
# 확인 : 0.4170 과 0.8021 이 나온다

# loss_right = loss_fn(logits3, target)                        # ○ 로짓을 그대로 넣는다
# loss_wrong = loss_fn(torch.softmax(logits3, dim=1), target)  # ✗ softmax 를 먼저 씌우면

print(f"○ logits 를 그대로     loss = {loss_right.item():.4f}")
print(f"✗ softmax 를 씌우면    loss = {loss_wrong.item():.4f}   ← 같은 예측인데 손실이 커졌다")

<details>
<summary>✅ <b>정답 보기</b> — 실습 4 — 주석 해제 (2줄)</summary>

```python
loss_right = loss_fn(logits3, target)                        # ○ 로짓을 그대로
loss_wrong = loss_fn(torch.softmax(logits3, dim=1), target)  # ✗ softmax 를 먼저 씌우면
```

맨 앞 `#` 과 뒤따르는 공백 한 칸만 지우면 됩니다. 결과는 **0.4170 / 0.8021**.

</details>

> **함정 1 — 정답은 원-핫이 아니라 정수** ★
> `target = torch.tensor([0])` 처럼 **클래스 번호**를 줍니다. `[1, 0, 0]` 이 아닙니다.
> 3교시 FashionMNIST 에서도 정답이 `3`, `7` 같은 숫자로 들어옵니다.
>
> **함정 2 — 모델 마지막에 Softmax 를 붙이지 않는다** ★ (중간고사 출제 지점)
> `CrossEntropyLoss` 는 **안에서 Softmax 를 이미 처리**합니다.
> 위에서 보았듯 따로 씌우면 **두 번 씌우는 것**이 되어 손실이 0.4170 → 0.8021 로 나빠집니다.
> 4주차의 `BCEWithLogitsLoss` 와 **정확히 같은 이야기**입니다.
>
> → **모델의 마지막 층은 `nn.Linear` 로 끝냅니다.** 확률을 사람이 보고 싶을 때만
> 따로 `torch.softmax` 를 씌웁니다 (3교시 실습 10).

---

## 1교시 핵심 정리와 체크리스트

- [ ] 4주차 학습 루프 **5단계**를 보지 않고 순서대로 말할 수 있다
- [ ] **층만 쌓고 활성함수를 빼면 왜 안 되는지** 실습 1의 숫자로 설명할 수 있다 ★★
- [ ] **MLP 의 구조**(784→256→128→10)와 은닉층 뒤에만 ReLU 를 붙이는 것을 안다
- [ ] ReLU · Sigmoid · Softmax 의 **모양**과 쓰는 자리를 안다
- [ ] **기울기를 곱하며 뒤로 간다**는 것과 `0.2×0.2×…=0.00032` 의 뜻을 안다 ★
- [ ] Sigmoid 를 깊게 쌓으면 **앞쪽 층 기울기가 작아지는 것**을 표로 확인했다 ★
- [ ] `CrossEntropyLoss` 의 함정 두 가지를 말할 수 있다 ★

### 저장하고 제출 준비

1. `Kernel → Restart Kernel and Run All Cells` 로 처음부터 다시 한 번 돌립니다
2. **출력이 살아 있는 상태로** `06_xor_and_activations.ipynb` 로 저장합니다
3. 2교시 노트북(`07_mlp_fashionmnist_blank.ipynb`)을 엽니다

> **다음 시간 예고**: 층과 활성함수는 정해졌습니다. 그런데 **돌리기 전에 맞춰야 할 조건**이
> 둘 있습니다 — 입력값의 크기와, 가중치의 **출발값**입니다.
> 2교시에는 그 둘을 맞추고, 모델을 클래스로 묶고, 6만 장을 흘려보낼 통로를 만듭니다.